# 12 Send a command to the edge

Phase 5. Sends **one operating command** to a turbine of Northwind Ridge: it writes the command to the eventstream `WindFarmCommands`, from where a data flow of Azure IoT Operations carries it to the control topic on the edge.

- The control topic takes five commands: `curtail`, `release`, `stop`, `start`, `reset`. It cannot inject a fault.
- As a job: the operations agent of Phase 5b runs this notebook as its action `Curtail turbine` and sets the four values of the first code cell: `turbine`, `percent`, `duration`, `requested_by`. A run that names no turbine is refused. The command is not among them: as a job this notebook curtails and does nothing else.
- By hand: set `turbine` in the first code cell, then **Run all**. The second code cell holds the command and `dry_run`; with `dry_run = True` the notebook builds and shows the command and sends nothing. For sending commands by hand as a habit there is `13_command_center`.
- Before the first run: `bash infra/08-create-command-path.sh` on the workstation (guide: `docs/12-deployment-guide-phase5a.md`).

The notebook reads where to write from the Fabric API each time it runs. No key is kept in it.

In [ ]:
# Parameters: what a job or the operations agent sets. These four and nothing else (this is the parameter cell).
turbine = ""               # WT-01 to WT-12. Empty on purpose: a command names its turbine
percent = 30               # the share of rated power the turbine may still produce
duration = "10m"           # how long the limit lasts: for example 90s, 10m, 1h
requested_by = ""          # who asked for it; empty means the user who runs the notebook

In [ ]:
# Not parameters, on purpose. This cell runs after a caller's values are set, so no caller can change
# what stands here: run as a job, this notebook curtails and does nothing else.
# By hand you may change these for one run. For other commands by hand, 13_command_center is the better place.
command = "curtail"        # curtail, release, stop, start, reset
reason = ""                # stop: operator or maintenance; otherwise a note that travels with the command
dry_run = False            # True: build and show the command, send nothing
EVENTSTREAM = "WindFarmCommands"

In [ ]:
# Generated by tools/build-phase5.py from tools/edge_command.py. Change it there.
import json
import re
import uuid
from datetime import datetime, timezone

COMMANDS = ("curtail", "release", "stop", "start", "reset")
TURBINE = re.compile(r"^WT-(0[1-9]|1[0-2])$")
DURATION = re.compile(r"^[1-9][0-9]{0,3}[smh]$")
# How a person or an agent may write a duration: 10m, 10 m, 10 min, 10 minutes, 90 seconds, 1 hour.
SPELLED_DURATION = re.compile(r"^([1-9][0-9]{0,3})\s*(s|sec|secs|second|seconds|m|min|mins|minute|minutes|h|hr|hrs|hour|hours)$")
STOP_REASONS = ("operator", "maintenance")


class CommandError(ValueError):
    """The command would not be accepted, or would do more than was asked for."""


def build_command(command, turbine, percent=None, duration="", reason="", requested_by="", request_id=None, now=None):
    """The JSON object for the control topic, after checking every field.

    The simulator treats a missing turbine as "all twelve". Here a turbine is always required, and
    all of them only when the word `all` is written out.
    """
    name = str(command or "").strip().lower()
    if name not in COMMANDS:
        raise CommandError(f"'{command}' is not a command of the control topic. Use one of: {', '.join(COMMANDS)}.")
    target = str(turbine or "").strip()
    if target.lower() == "all":
        target = "all"
    else:
        target = target.upper()
        if not TURBINE.match(target):
            raise CommandError(f"'{turbine}' is not a turbine. Use WT-01 to WT-12, or the word all.")
    out = {"command": name, "requestId": request_id or uuid.uuid4().hex[:12], "turbine": target}

    text = str(duration or "").strip().lower()
    if text and name not in ("curtail", "stop"):
        raise CommandError(f"'{name}' takes no duration.")
    if text and not DURATION.match(text):
        raise CommandError(f"'{duration}' is not a duration. Write a number and s, m or h, for example 30m.")
    if name == "curtail":
        try:
            value = float(percent)
        except (TypeError, ValueError):
            raise CommandError("curtail needs percent: the share of rated power the turbine may still produce, 0 to 100.") from None
        if not 0 <= value <= 100:
            raise CommandError(f"percent is {value:g}; it must be between 0 and 100.")
        if not text:
            raise CommandError("curtail needs a duration here, for example 30m, so that a forgotten command ends by itself. To end it sooner, send release.")
        out["percent"] = value
    elif percent not in (None, ""):
        raise CommandError(f"'{name}' takes no percent.")
    if text:
        out["duration"] = text
    if name == "stop":
        why = str(reason or "operator").strip().lower()
        if why not in STOP_REASONS:
            raise CommandError(f"The reason of a stop is one of: {', '.join(STOP_REASONS)}.")
        out["reason"] = why
    elif reason:
        out["note"] = str(reason)[:200]
    if requested_by:
        out["requestedBy"] = str(requested_by)[:120]
    out["issuedAt"] = (now or datetime.now(timezone.utc)).strftime("%Y-%m-%dT%H:%M:%SZ")
    return out


def as_flag(value, name="dry_run"):
    """True or False from a parameter that may arrive as text. The text "False" is not a true value."""
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in ("true", "1", "yes"):
        return True
    if text in ("false", "0", "no", ""):
        return False
    raise CommandError(f"{name} is '{value}'; write True or False.")


def from_parameters(command, turbine, percent=None, duration="", reason="", requested_by="", request_id=None, now=None):
    """The command for the values of the notebook's parameter cell, however they were written.

    Whoever fills that cell, a person, a pipeline or the operations agent, changes the values the
    command needs and leaves the others as they stand, and may write `Curtail`, `30%` or `10 minutes`.
    So the name is brought to lower case before anything is decided by it, a value the command does
    not take is left out, and a percentage or a duration is read in the forms people write them.
    Deciding by the name as it was typed dropped the duration of a `Stop`, which then had no end.
    """
    name = str(command or "").strip().lower()
    if name == "curtail" and isinstance(percent, str):
        percent = percent.strip().rstrip("%").strip()
    text = str(duration or "").strip().lower()
    spelled = SPELLED_DURATION.match(text)
    if spelled:
        text = spelled.group(1) + spelled.group(2)[0]
    return build_command(name, turbine, percent=percent if name == "curtail" else None,
                         duration=text if name in ("curtail", "stop") else "", reason=reason,
                         requested_by=requested_by, request_id=request_id, now=now)


def find_item(items, name):
    """The id of the item of that display name in the answer of a Fabric list call, or None."""
    return next((item.get("id") for item in items if item.get("displayName") == name), None)


def find_custom_endpoint(topology, kind):
    """The id of the custom endpoint among the `sources` or the `destinations` of an eventstream, or None."""
    return next((node.get("id") for node in topology.get(kind, []) if node.get("type") == "CustomEndpoint"), None)


def send_settings(connection):
    """What a sender needs from the connection of a custom endpoint source, checked for completeness."""
    namespace = connection.get("fullyQualifiedNamespace") or ""
    event_hub = connection.get("eventHubName") or ""
    secret = (connection.get("accessKeys") or {}).get("primaryConnectionString") or ""
    missing = [label for label, value in (("fullyQualifiedNamespace", namespace), ("eventHubName", event_hub), ("accessKeys.primaryConnectionString", secret)) if not value]
    if missing:
        raise CommandError("The connection of the eventstream's source lacks: " + ", ".join(missing) + ".")
    return {"namespace": namespace, "eventHub": event_hub, "connectionString": secret}


def payload(command):
    """The bytes that go on the wire: compact JSON, as the simulator's own tool writes it."""
    return json.dumps(command, separators=(",", ":")).encode("utf-8")

In [ ]:
# The command, checked before anything is installed or sent.
if not requested_by:
    try:
        requested_by = notebookutils.runtime.context.get("userName") or ""
    except Exception:
        requested_by = ""
dry_run = as_flag(dry_run)
cmd = from_parameters(command, turbine, percent=percent, duration=duration, reason=reason, requested_by=requested_by)
if cmd["turbine"] == "all":
    raise CommandError("This notebook acts on one turbine at a time. For all twelve at once, send from 13_command_center.")
print(json.dumps(cmd, indent=1))

In [ ]:
# Where to write: the custom endpoint that is the source of the eventstream, asked from the Fabric API.
import sempy.fabric as fabric

client = fabric.FabricRestClient()
workspace_id = fabric.get_workspace_id()


def ask(path):
    response = client.get(path)
    if response.status_code != 200:
        raise SystemExit(f"The Fabric API answered {response.status_code} to {path}: {response.text[:300]}")
    return response.json()


eventstream_id = find_item(ask(f"v1/workspaces/{workspace_id}/eventstreams").get("value", []), EVENTSTREAM)
if not eventstream_id:
    raise SystemExit(f"No eventstream named {EVENTSTREAM} in this workspace. Run: bash infra/08-create-command-path.sh")
topology = ask(f"v1/workspaces/{workspace_id}/eventstreams/{eventstream_id}/topology")
source_id = find_custom_endpoint(topology, "sources")
if not source_id:
    raise SystemExit(f"The eventstream {EVENTSTREAM} has no custom endpoint as a source. Add one in the portal and publish.")
settings = send_settings(ask(f"v1/workspaces/{workspace_id}/eventstreams/{eventstream_id}/sources/{source_id}/connection"))
print(f"Eventstream {EVENTSTREAM} ({eventstream_id})")
print(f"Writes to {settings['namespace']}, event hub {settings['eventHub']}")

In [ ]:
# Send. One event, over port 443.
def bring(module, pin):
    """Import a library, and install it first when this session does not have it.

    pip runs as a plain command here, not as the %pip magic: the magic is switched off when a notebook
    runs as a job, which is how the operations agent runs this one."""
    import importlib, site, subprocess, sys
    try:
        return importlib.import_module(module)
    except ImportError:
        done = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", pin],
                              capture_output=True, text=True)
        if done.returncode != 0:
            raise SystemExit(f"pip could not install {pin}: {(done.stderr or done.stdout)[-600:]}")
        if site.getusersitepackages() not in sys.path:   # where pip installs when the runtime's own folder is read-only
            sys.path.append(site.getusersitepackages())
        importlib.invalidate_caches()
        print(f"Installed {pin} for this session.")
        return importlib.import_module(module)


if dry_run:
    print("dry_run is True: nothing was sent.")
else:
    bring("websocket", "websocket-client==1.9.2")   # the transport over port 443; azure-eventhub does not bring it
    bring("azure.eventhub", "azure-eventhub==5.15.1")
    from azure.eventhub import EventData, EventHubProducerClient, TransportType

    extra = {} if "EntityPath=" in settings["connectionString"] else {"eventhub_name": settings["eventHub"]}
    producer = EventHubProducerClient.from_connection_string(
        settings["connectionString"], transport_type=TransportType.AmqpOverWebsocket, **extra)
    with producer:
        event = EventData(payload(cmd))
        event.content_type = "application/json"
        batch = producer.create_batch()
        batch.add(event)
        producer.send_batch(batch)
    print(f"Sent: {cmd['command']} for {cmd['turbine']}, request {cmd['requestId']}, at {cmd['issuedAt']} UTC.")
    print("The turbine should follow within a minute. Check with the query in the next cell.")

## Check

In a KQL queryset on `WindFarmEH`, with the turbine of the command:

```kusto
TurbineTelemetry
| where TurbineId == 'WT-07' and Timestamp > ago(5m)
| project Timestamp, State, PowerSetpoint, ActivePower
| top 5 by Timestamp desc
```

After `curtail` to 30 percent: `State` is `CURTAILED` and `PowerSetpoint` is 1260 (30 percent of 4,200 kW), within a minute. In the twin the ring of the turbine turns blue. On the edge, `bash infra/vm.sh ssh bash ./edge/scripts/simctl.sh status` shows the same.

If nothing happens, the guide says where to look: the eventstream's own view first, then the data flow on the edge.